In [1]:
from Bio import SeqIO
from Bio.SeqUtils.ProtParam import ProteinAnalysis as PA
from modlamp.descriptors import PeptideDescriptor, GlobalDescriptor
from sklearn.model_selection import train_test_split,KFold
import pandas as pd
import os, re, math, platform
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from numpy import linalg as la
import os
import random
import warnings
warnings.filterwarnings('ignore')
import random
from fea_extract import read_fasta,insert_AAC,insert_DPC,insert_CKSAAGP,insert_CTD,insert_PAAC,insert_AAI,insert_GTPC,insert_QSO,insert_AAE,insert_PSAAC,insert_word2int,insert_ASDC
from tools import del_data
seed = 10
from pathlib import Path
Path('./results/process_data/').mkdir(exist_ok=True,parents=True)
Path('./results/balance/').mkdir(exist_ok=True,parents=True)

In [3]:
# Parsing a fasta or csv file
def del_data(inFile):
    seq = read_fasta(inFile)
    seqname = seq.to_numpy()
    newseq = []
    j = seqname.shape[0]
    for i in range(j):
        if 6 <= len(seqname[i][1])<=100:
            newseq.append(seqname[i])
    newseq = np.array(newseq)
    print('shape of data：', newseq.shape)
    newseq = pd.DataFrame(data=newseq, columns=["Id", "Sequence"])
    return newseq

In [6]:
#now combine previosu training, generated and validation dataset to create new anticancer and antimicrobial peptide dataset
def combine_fasta_files(input_files, output_file):
    combined_seqs = {}
    
    for file in input_files:
        with open(file, 'r') as f:
            lines = f.readlines()
            header = None
            sequence = ''
            for line in lines:
                if line.startswith('>'):
                    if header is not None:
                        combined_seqs[header] = sequence
                    header = line.strip()
                    sequence = ''
                else:
                    sequence += line.strip()
            if header is not None:
                combined_seqs[header] = sequence

    with open(output_file, 'w') as out:
        for header, sequence in combined_seqs.items():
            out.write(f"{header}\n{sequence}\n")

# Example Usage:
input_files = ['AMP_GEN_300.fasta', 'amp_train_neg.fasta', 'amp_val_neg.fasta']
output_file = 'AMP.fasta'

combine_fasta_files(input_files, output_file)

In [7]:
seq_ACP =del_data('data/ACP.fasta')
seq_AMP =del_data('data/AMP.fasta')

shape of data： (1368, 2)
shape of data： (1094, 2)


In [8]:
from tools import pro_data

In [11]:
#Splitting a data to create a train and test data from this orginal dataset  #80%，20%
prefix = [seq_ACP,seq_AMP]
Prefix = ['seq_ACP','seq_AMP']
for i in range(2):
    df_train,df_test = train_test_split(prefix[i],random_state=seed,test_size=.2)
    df_train.to_csv(os.path.join( "data/train/{:s}_train.csv".format(Prefix[i])), index=False)
    df_test.to_csv(os.path.join("data/test/{:s}_test.csv".format(Prefix[i])), index=False)
    print("Done!")

Done!
Done!


In [13]:
#saving file to csv
train_sets = {
    lab:pd.read_csv('data/train/{:s}_train.csv'.format(lab))
    for lab in ['seq_ACP','seq_AMP']
}
test_sets = {
    lab:pd.read_csv('data/test/{:s}_test.csv'.format(lab))
    for lab in ['seq_ACP','seq_AMP']
}
train_sets['seq_ACP'].loc[:, 'Label'] = 1 #set label for classes
train_sets['seq_AMP'].loc[:, 'Label'] = 0 #set label for classes
test_sets['seq_ACP'].loc[:, 'Label'] = 1
test_sets['seq_AMP'].loc[:, 'Label'] = 0

#create a dataset, train and test dataset
all_train = pd.concat([train_sets['seq_ACP'],train_sets['seq_AMP']],ignore_index='ignore')
all_test = pd.concat([test_sets['seq_ACP'],test_sets['seq_AMP']],ignore_index='ignore')
X_train = all_train.iloc[:,0:2]
X_test = all_test.iloc[:,0:2]
y_train = all_train['Label']
y_test = all_test["Label"]
X_test.to_csv('data/test/X_test.csv',index = False)
y_test.to_csv('data/test/y_test.csv',index = False)
X_train.to_csv('data/train/X_train.csv',index = False)
y_train.to_csv('data/train/y_train.csv',index = False)

In [14]:
#define a encoding for training
insert_list=[insert_AAC,insert_DPC,insert_CKSAAGP,insert_PSAAC,insert_CTD,insert_GTPC,
             insert_QSO,insert_AAE,insert_AAI,insert_ASDC,insert_PAAC,pro_data]
insert_str=["insert_AAC","insert_DPC","insert_CKSAAGP","insert_PSAAC","insert_CTD",
            "insert_GTPC","insert_QSO","insert_AAE","insert_AAI","insert_ASDC","insert_PAAC",'insert_All_data']

In [18]:
n = len(insert_list)
for j in range(n):
    print("encoding{}_{}".format(str(j),insert_str[j][7:]))
    X_test = pd.read_csv('data/test/X_test.csv')
    X_train = pd.read_csv('data/train/X_train.csv')
    df_seq_train = insert_list[j](X_train)
    df_seq_train.to_csv('results/process_data/train_{}.csv'.format(insert_str[j][7:]),index =False )
    df_seq_test = insert_list[j](X_test)
    df_seq_test.to_csv('results/process_data/test_{}.csv'.format(insert_str[j][7:]),index =False )

encoding0_AAC
encoding1_DPC
encoding2_CKSAAGP
encoding3_PSAAC
encoding4_CTD
encoding5_GTPC
encoding6_QSO
encoding7_AAE
encoding8_AAI
encoding9_ASDC
encoding10_PAAC
encoding11_All_data


In [19]:
from tools import padseq
X_test = pd.read_csv('data/test/X_test.csv')
X_train = pd.read_csv('data/train/X_train.csv')
new_seq_train = padseq(X_train)
new_seq_test = padseq(X_test)
word2_seq_train = insert_word2int(new_seq_train)
word2_seq_train.to_csv('results/process_data/train_word2int.csv',index=False)

word2_seq_test = insert_word2int(new_seq_test)
word2_seq_test.to_csv('results/process_data/test_word2int.csv',index=False)

In [20]:
insert_list=[insert_AAC,insert_DPC,insert_CKSAAGP,insert_PSAAC,insert_CTD,insert_GTPC,
             insert_QSO,insert_AAE,insert_AAI,insert_ASDC,insert_PAAC]
insert_str=["insert_AAC","insert_DPC","insert_CKSAAGP","insert_PSAAC","insert_CTD",
            "insert_GTPC","insert_QSO","insert_AAE","insert_AAI","insert_ASDC","insert_PAAC"]